In [ ]:
import os
import numpy as np
from json_tricks import dump, load

from pydub import AudioSegment, effects
import librosa
import librosa.display
import noisereduce as nr
from noisereduce.noisereducev1 import reduce_noise

#final_x = reduce_noise(audio_clip=padded_x, 
                          #noise_clip=padded_x, 
                          #verbose=False)


import matplotlib.pyplot as plt
from librosa import display   
import IPython.display as ipd


import tensorflow as tf
import keras
import sklearn

In [1]:
import os
import numpy as np
from json_tricks import dump, load

from pydub import AudioSegment, effects
import librosa
import librosa.display
import noisereduce as nr
from noisereduce.noisereducev1 import reduce_noise

#final_x = reduce_noise(audio_clip=padded_x, 
                          #noise_clip=padded_x, 
                          #verbose=False)


import matplotlib.pyplot as plt
from librosa import display   
import IPython.display as ipd


import tensorflow as tf
import keras
import sklearn

C:\Users\Ryan.Donovan\Anaconda3\envs\audio_test\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
C:\Users\Ryan.Donovan\Anaconda3\envs\audio_test\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Emotion kind validation function for TESS database, due to emotions written within the file name

def find_emotion_T(name):
    if('neutral' in name): return "01"
    elif('happy' in name): return "03"
    elif('sad' in name): return "04"
    elif('angry' in name): return "05"
    elif('fear' in name): return "06"
    elif('disgust' in name): return "07"
    elif('ps' in name): return "08"
    else: return "-1"

# 'emotions' list fix for classification purposes:
#     Classification values start from 0, Thus an 'n = n-1' operation has been executed for both RAVDESS and TESS databases:
def emotionfix(e_num):
    if e_num == "01": return 0 # neutral
    elif e_num == "02": return 1 #calm
    elif e_num == "03": return 2 #happy
    elif e_num == "04": return 3 #sad
    elif e_num == "05": return 4 #angry
    elif e_num == "06": return 5 #fear
    elif e_num == "07": return 6 #disgust
    else:               return 7 #surprised





In [3]:
# Maximum samples count for padding purposes.

sample_lengths = []
folder_path = r'C:\Users\Ryan.Donovan\Desktop\PhD\PhD Data\Audio\AudioFiles'

for subdir, dirs, files in os.walk(folder_path):
    for file in files:
        x, sr = librosa.load(path = os.path.join(subdir, file), sr = None)
        xt, index = librosa.effects.trim(x, top_db = 30)

        sample_lengths.append(len(xt))

print('Maxium sample length:', np.max(sample_lengths))


Maxium sample length: 204288


In [ ]:
import time

tic = time.perf_counter()

#initalise data lists

rms = []
zcr = []
mfcc = []
emotions = []

# Initialize variables
total_length = 204288 # desired frame length for all of the audio samples.
frame_length = 2048
hop_length = 512

folder_path = r'C:\Users\Ryan.Donovan\Desktop\PhD\PhD Data\Audio\AudioFiles'

for subdir, dirs, files in os.walk(folder_path):
    for file in files:

        # Fetch the sample rate.
        _, sr = librosa.load(path = os.path.join(subdir, file), sr = None)
        # Load the audio file
        rawsound = AudioSegment.from_file(os.path.join(subdir, file))
        # Normalize the Audio to +5.0 DdBFS.
        normalizedsound = effects.normalize(rawsound, headroom = 0)
        # Transform the normalized audio to np.array to samples.
        normal_x = np.array(normalizedsound.get_array_of_samples(), dtype = 'float32')
        # Trim silence from the beginning and the end.
        xt, index = librosa.effects.trim(normal_x, top_db = 30)
        #print(file,"\t", len(xt), "\t", rawsound.dBFS, "\t", normalizedsound.dBFS) #--QA purposes if needed-- 
        # Pad for duration equalization.
        #padded_x = np.pad(xt, (0, total_length-len(xt)), 'constant')
        # Noise Reduction
        #OLD WAY
        #final_x = reduce_noise(audio_clip=padded_x, noise_clip=padded_x, verbose=False)
        #Update
        final_x = nr.reduce_noise(normal_x, sr=sr)

        #Features Extraction
        f1 = librosa.feature.rms(final_x, frame_length = frame_length, hop_length = hop_length)
        f2 = librosa.feature.zero_crossing_rate(final_x , frame_length=frame_length, hop_length=hop_length, center=True) # ZCR      
        f3 = librosa.feature.mfcc(final_x, sr=sr, n_mfcc=13, hop_length = hop_length) # MFCC

        # Emotion Extraction from the Different Databases

        if(find_emotion_T(file) != "-1"): #TESS Database Validation
            name = find_emotion_T(file)
        else:
            name = file[6:8]

        #Filling the data lists
        rms.append(f1)
        zcr.append(f2)
        mfcc.append(f3)
        emotions.append(emotionfix(name))


toc = time.perf_counter()
print(f"Running time: {(toc - tic)/60:0.4f} minutes")


        
    


In [25]:
# Adjusting features shape to the 3D format: (batch, timesteps, feature)

f_rms = np.asarray(rms).astype('float32')
f_rms = np.swapaxes(f_rms,1,2)
f_zcr = np.asarray(zcr).astype('float32')
f_zcr = np.swapaxes(f_zcr,1,2)
f_mfccs = np.asarray(mfcc).astype('float32')
f_mfccs = np.swapaxes(f_mfccs,1,2)

print('ZCR shape:',f_zcr.shape)
print('RMS shape:',f_rms.shape)
print('MFCCs shape:',f_mfccs.shape)

ZCR shape: (4240, 497, 1)
RMS shape: (4240, 497, 1)
MFCCs shape: (4240, 497, 13)


In [26]:
# Concatenating all features to 'X' variable.
X = np.concatenate((f_zcr, f_rms, f_mfccs), axis=2)

# Preparing 'Y' as a 2D shaped variable.
Y = np.asarray(emotions).astype('int8')
Y = np.expand_dims(Y, axis=1)

In [ ]:
# Save X,Y arrays as lists to json files.

x_data = X.tolist() 
x_path = r'C:\Users\Ryan.Donovan\Desktop\PhD\PhD Data\Audio\src\dataX_datanew.json' # FILE SAVE PATH
dump(obj = x_data, fp = x_path)

y_data = Y.tolist() 
y_path = r'C:\Users\Ryan.Donovan\Desktop\PhD\PhD Data\Audio\src\Y_datanew.json' # FILE SAVE PATH
dump(obj = y_data, fp = y_path)

In [29]:
# Load X,Y json files back into lists, convert to np.arrays

x_path = r'C:\Users\Ryan.Donovan\Desktop\PhD\PhD Data\Audio\src\dataX_datanew.json' # FILE LOAD PATH
X = load(x_path)
X = np.asarray(X, dtype = 'float32')

y_path = r'C:\Users\Ryan.Donovan\Desktop\PhD\PhD Data\Audio\src\Y_datanew.json' # FILE LOAD PATH
Y = load(y_path)
Y = np.asarray(Y, dtype = 'int8')

In [ ]:
import time

tic = time.perf_counter()

#initalise data lists

rms = []
zcr = []
mfcc = []
emotions = []

# Initialize variables
total_length = 254288 # desired frame length for all of the audio samples.
frame_length = 2048
hop_length = 512

folder_path = r'C:\Users\Ryan.Donovan\Desktop\PhD\PhD Data\Audio\AudioFiles'

for subdir, dirs, files in os.walk(folder_path):
    for file in files:

        # Fetch the sample rate.
        _, sr = librosa.load(path = os.path.join(subdir, file), sr = None)
        # Load the audio file
        rawsound = AudioSegment.from_file(os.path.join(subdir, file))
        # Normalize the Audio to +5.0 DdBFS.
        normalizedsound = effects.normalize(rawsound, headroom = 0)
        # Transform the normalized audio to np.array to samples.
        normal_x = np.array(normalizedsound.get_array_of_samples(), dtype = 'float32')
        # Trim silence from the beginning and the end.
        xt, index = librosa.effects.trim(normal_x, top_db = 30)
        #print(file,"\t", len(xt), "\t", rawsound.dBFS, "\t", normalizedsound.dBFS) #--QA purposes if needed-- 
        # Pad for duration equalization.
        print(len(xt))
        padded_x = np.pad(xt, (0, total_length-len(xt)), 'constant')
        # Noise Reduction
        #OLD WAY
        #final_x = reduce_noise(audio_clip=padded_x, noise_clip=padded_x, verbose=False)
        #Update
        final_x = nr.reduce_noise(padded_x, sr=sr)

        #Features Extraction
        f1 = librosa.feature.rms(final_x, frame_length = frame_length, hop_length = hop_length)
        f2 = librosa.feature.zero_crossing_rate(final_x , frame_length=frame_length, hop_length=hop_length, center=True) # ZCR      
        f3 = librosa.feature.mfcc(final_x, sr=sr, n_mfcc=13, hop_length = hop_length) # MFCC

        # Emotion Extraction from the Different Databases

        if(find_emotion_T(file) != "-1"): #TESS Database Validation
            name = find_emotion_T(file)
        else:
            name = file[6:8]

        #Filling the data lists
        rms.append(f1)
        zcr.append(f2)
        mfcc.append(f3)
        emotions.append(emotionfix(name))


toc = time.perf_counter()
print(f"Running time: {(toc - tic)/60:0.4f} minutes")


        
    


In [30]:
# Split to train, validation, and test sets.
from sklearn.model_selection import train_test_split
x_train, x_tosplit, y_train, y_tosplit = train_test_split(X, Y, test_size = 0.125, random_state = 1)
x_val, x_test, y_val, y_test = train_test_split(x_tosplit, y_tosplit, test_size = 0.304, random_state = 1)

#'One-hot' vectors for Y: emotion classification
y_train_class = tf.keras.utils.to_categorical(y_train, 8, dtype = 'int8')
y_val_class = tf.keras.utils.to_categorical(y_val, 8, dtype = 'int8')

In [31]:
# x_train, x_val, and x_test shape check.
print(np.shape(x_train))
print(np.shape(x_val))
print(np.shape(x_test))

(3710, 497, 15)
(368, 497, 15)
(162, 497, 15)


In [32]:
# Save x_test, y_test to JSON.

file_path = 'x_test_data.json'
dump(obj = x_test, fp = file_path)

file_path = 'y_test_data.json'
dump(obj = y_test, fp = file_path)

'{"__ndarray__": [[6], [3], [5], [2], [2], [7], [0], [1], [2], [6], [4], [7], [0], [6], [6], [4], [2], [7], [7], [4], [3], [4], [4], [6], [3], [2], [4], [6], [3], [2], [6], [2], [3], [4], [4], [2], [7], [7], [4], [5], [5], [3], [6], [7], [1], [2], [3], [7], [4], [3], [6], [0], [6], [6], [2], [0], [5], [6], [6], [4], [3], [1], [0], [7], [3], [6], [0], [3], [4], [5], [5], [6], [2], [4], [5], [2], [2], [7], [6], [0], [5], [6], [5], [6], [3], [3], [4], [0], [2], [4], [7], [6], [7], [6], [6], [3], [4], [2], [4], [5], [6], [3], [4], [7], [4], [2], [3], [3], [2], [2], [7], [4], [5], [2], [6], [2], [6], [2], [2], [5], [5], [4], [3], [3], [5], [7], [0], [2], [4], [7], [6], [5], [0], [1], [5], [2], [4], [3], [2], [5], [2], [0], [0], [2], [1], [4], [0], [2], [3], [2], [7], [5], [2], [6], [0], [2], [3], [7], [4], [0], [5], [3]], "dtype": "int8", "shape": [162, 1], "Corder": true}'

In [33]:
from keras.models import Sequential
from keras import layers
from keras import optimizers
from keras import callbacks 

In [ ]:
# Initializing the model

model = Sequential()
model.add(layers.LSTM(64, return_sequences = True, input_shape=(X.shape[1:3])))
model.add(layers.LSTM(64))
model.add(layers.Dense(8, activation = 'softmax'))
print(model.summary())

batch_size = 23

# Callbacks functions
checkpoint_path = r'C:\Users\Ryan.Donovan\Desktop\PhD\PhD Data\Audio\src\best_weights.hdf5'

#-> Save the best weights
mcp_save = callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True,
                           monitor='val_categorical_accuracy',
                           mode='max')
#-> Reduce learning rate after 100 epoches without improvement.
rlrop = callbacks.ReduceLROnPlateau(monitor='val_categorical_accuracy', 
                                    factor=0.1, patience=100)
                             
# Compile & train   
model.compile(loss='categorical_crossentropy', 
                optimizer='RMSProp', 
                metrics=['categorical_accuracy'])

history = model.fit(x_train, y_train_class, 
                      epochs=340, batch_size = batch_size, 
                      validation_data = (x_val, y_val_class), 
                      callbacks = [mcp_save, rlrop])
# Define the best weights to the model.
model.load_weights(checkpoint_path)